## Investigating The Performance of the S&P500

In [9]:
import pandas as pd

In [59]:
sp500 = pd.read_csv('../data/indices/yahoo/gspc.csv', parse_dates=True, index_col=0).sort_index(ascending=True)
sp500

,Open,High,Low,Close (split adjusted),Adj Close(dividends and splits),Volume
Date,,,,,,
2018-02-28,2753.78,2761.52,2713.54,2713.83,2713.83,4230660000
2018-03-01,2715.22,2730.89,2659.65,2677.67,2677.67,4503970000
2018-03-02,2658.89,2696.25,2647.32,2691.25,2691.25,3882450000
2018-03-05,2681.06,2728.09,2675.75,2720.94,2720.94,3710810000
2018-03-06,2730.18,2732.08,2711.26,2728.12,2728.12,3370690000
...,...,...,...,...,...,...
2021-04-12,4124.71,4131.76,4114.82,4127.99,4127.99,3578500000
2021-04-13,4130.10,4148.00,4124.43,4141.59,4141.59,3728440000
2021-04-14,4141.58,4151.69,4120.87,4124.66,4124.66,3976540000


# Goals

I need to have data that is useful when analysing the news sentiment. It would be productive to offer a module that would provide data on:

* Returns 
    * Look at the % between adj close of two days
    * Difference from adj close to the open price
    
* Market Action
    * Volatility
    * Volume
    * Trading range
    
* Drawdowns
    * Look at investment_management/1/103
    
* Other statistical measures
    * https://docs.scipy.org/doc/scipy/reference/stats.html (or the later courses in investment management)
    

Through doing this I should also think about how (if) I take the effect of a weekend into account. Can take a look at if returns on Monday are different enough.

### Returns

In [13]:
returns = sp500[['Open', 'Adj Close(dividends and splits)']]
returns['Change'] = sp500[['Adj Close(dividends and splits)']].pct_change()
returns

<ipython-input-13-aaf3d9d00e18>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  returns['Change'] = sp500[['Adj Close(dividends and splits)']].pct_change()


,Open,Adj Close(dividends and splits),Change
Date,,,
2018-02-28,2753.78,2713.83,NaN
2018-03-01,2715.22,2677.67,-0.013324
2018-03-02,2658.89,2691.25,0.005072
2018-03-05,2681.06,2720.94,0.011032
2018-03-06,2730.18,2728.12,0.002639
...,...,...,...
2021-04-12,4124.71,4127.99,-0.000196
2021-04-13,4130.10,4141.59,0.003295
2021-04-14,4141.58,4124.66,-0.004088


In [25]:
# Not sure how to do this nicely in pandas, pretty hacky
import numpy as np

close_to_open = []
close_to_open.append(np.nan)

for i in range (1, len(returns)):
    previous_close = returns.iloc[int(i) - 1]['Adj Close(dividends and splits)']
    open_price = returns.iloc[i]['Open']
    close_to_open.append((open_price-previous_close)/previous_close)

In [26]:
returns['Close to Open Change'] = close_to_open

<ipython-input-26-38268947712f>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  returns['Close to Open Change'] = close_to_open


In [27]:
returns

,Open,Adj Close(dividends and splits),Change,Close to Open Change
Date,,,,
2018-02-28,2753.78,2713.83,NaN,NaN
2018-03-01,2715.22,2677.67,-0.013324,0.000512
2018-03-02,2658.89,2691.25,0.005072,-0.007014
2018-03-05,2681.06,2720.94,0.011032,-0.003786
2018-03-06,2730.18,2728.12,0.002639,0.003396
...,...,...,...,...
2021-04-12,4124.71,4127.99,-0.000196,-0.000991
2021-04-13,4130.10,4141.59,0.003295,0.000511
2021-04-14,4141.58,4124.66,-0.004088,-0.000002


In [33]:
returns = returns[1:]
returns

,Open,Adj Close(dividends and splits),Change,Close to Open Change
Date,,,,
2018-03-01,2715.22,2677.67,-0.013324,0.000512
2018-03-02,2658.89,2691.25,0.005072,-0.007014
2018-03-05,2681.06,2720.94,0.011032,-0.003786
2018-03-06,2730.18,2728.12,0.002639,0.003396
2018-03-07,2710.18,2726.80,-0.000484,-0.006576
...,...,...,...,...
2021-04-12,4124.71,4127.99,-0.000196,-0.000991
2021-04-13,4130.10,4141.59,0.003295,0.000511
2021-04-14,4141.58,4124.66,-0.004088,-0.000002


In [36]:
returns.to_csv('../data/calculations/returns.csv')

### Market Action

In [37]:
sp500.columns

Index(['Open', 'High', 'Low', 'Close (split adjusted)',
       'Adj Close(dividends and splits)', 'Volume'],
      dtype='object')

#### Volatility
I was thinking about taking the volatility of the entire period by I did'nt want the look-ahead error.

In [61]:
rolling_30_avg = sp500['Adj Close(dividends and splits)'].rolling(30).mean()
rolling_30_avg

Date
2018-02-28            NaN
2018-03-01            NaN
2018-03-02            NaN
2018-03-05            NaN
2018-03-06            NaN
                 ...     
2021-04-12    3951.577333
2021-04-13    3959.569667
2021-04-14    3968.048667
2021-04-15    3979.738667
2021-04-16    3993.638667
Name: Adj Close(dividends and splits), Length: 789, dtype: float64

In [74]:
market_action = sp500[['Volume','Adj Close(dividends and splits)']]
market_action['30D avg price'] = rolling_30_avg
market_action

<ipython-input-74-e6cdfa0e68cb>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  market_action['30D avg price'] = rolling_30_avg


,Volume,Adj Close(dividends and splits),30D avg price
Date,,,
2018-02-28,4230660000,2713.83,NaN
2018-03-01,4503970000,2677.67,NaN
2018-03-02,3882450000,2691.25,NaN
2018-03-05,3710810000,2720.94,NaN
2018-03-06,3370690000,2728.12,NaN
...,...,...,...
2021-04-12,3578500000,4127.99,3951.577333
2021-04-13,3728440000,4141.59,3959.569667
2021-04-14,3976540000,4124.66,3968.048667


In [81]:
deviation = market_action['30D avg price'] - market_action['Adj Close(dividends and splits)']
market_action['Deviation'] = deviation

In [84]:
market_action['Deviation^2'] = deviation ** 2

In [86]:
market_action['30D Variance'] = market_action['Deviation^2'].rolling(30).mean()

In [87]:
market_action['Std Deviation'] = market_action['30D Variance'] ** 0.5

In [93]:
market_action.head(60)

,Volume,Adj Close(dividends and splits),30D avg price,Deviation,Deviation^2,30D Variance,Std Deviation
Date,,,,,,,
2018-02-28,4230660000,2713.83,NaN,NaN,NaN,NaN,NaN
2018-03-01,4503970000,2677.67,NaN,NaN,NaN,NaN,NaN
2018-03-02,3882450000,2691.25,NaN,NaN,NaN,NaN,NaN
2018-03-05,3710810000,2720.94,NaN,NaN,NaN,NaN,NaN
2018-03-06,3370690000,2728.12,NaN,NaN,NaN,NaN,NaN
2018-03-07,3393270000,2726.80,NaN,NaN,NaN,NaN,NaN
2018-03-08,3212320000,2738.97,NaN,NaN,NaN,NaN,NaN
2018-03-09,3364100000,2786.57,NaN,NaN,NaN,NaN,NaN
2018-03-12,3185020000,2783.02,NaN,NaN,NaN,NaN,NaN


In [94]:
market_action['2020-03-01':'2020-03-31']

,Volume,Adj Close(dividends and splits),30D avg price,Deviation,Deviation^2,30D Variance,Std Deviation
Date,,,,,,,
2020-03-02,6376400000,3090.23,3275.160667,184.930667,34199.351474,13845.397852,117.666469
2020-03-03,6355940000,3003.37,3264.285667,260.915667,68076.985112,15730.513304,125.421343
2020-03-04,5035480000,3130.12,3257.930000,127.810000,16335.396100,15994.491703,126.469331
2020-03-05,5575550000,3023.94,3248.003000,224.063000,50204.227969,17416.606624,131.971992
2020-03-06,6552140000,2972.37,3236.230667,263.860667,69622.451414,19500.425169,139.643923
2020-03-09,8423050000,2746.56,3217.933667,471.373667,222193.133627,26827.459933,163.790903
2020-03-10,7635960000,2882.23,3205.887000,323.657000,104753.853649,30317.875462,174.120290
2020-03-11,7374110000,2741.38,3188.058333,446.678333,199521.533469,36951.586488,192.227954
2020-03-12,8829380000,2480.64,3161.633000,680.993000,463751.466049,52401.154685,228.912985
